# 🧠 Denoising Autoencoder on MNIST
### Week 6 Assessment | Deep Learning Project

---

**Objective:**  
Build a deep learning model that can **remove noise from images** using a **Convolutional Autoencoder** trained on the MNIST handwritten-digits dataset.

**Approach:**
- Corrupt clean MNIST images by adding Gaussian noise
- Train an autoencoder to reconstruct the original clean image from the noisy version
- Evaluate qualitative (visual) and quantitative (MSE / SSIM) reconstruction quality

**Architecture Overview:**
```
Noisy Image ──► [Encoder: Conv layers + MaxPool] ──► Latent Representation
                                                              │
Clean Image ◄── [Decoder: ConvTranspose layers]  ◄──────────┘
```

> **Dataset:** [MNIST on Kaggle](https://www.kaggle.com/datasets/awsaf49/mnist-dataset) | **Reference:** [MNIST Autoencoder GitHub](https://github.com/NvsYashwanth/MNIST-AutoecncoderWeek6)


## 1. Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import MNIST

from sklearn.metrics import mean_squared_error
from skimage.metrics import structural_similarity as ssim

# ── reproducibility ──
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {DEVICE}")
print(f"✅ PyTorch version: {torch.__version__}")


## 2. Hyperparameters & Configuration

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
BATCH_SIZE   = 128
EPOCHS       = 20
LEARNING_RATE = 1e-3
NOISE_FACTOR  = 0.4       # std of Gaussian noise added to inputs
LATENT_DIM    = 128       # bottleneck channels (not used directly but sets encoder depth)

print("Hyperparameter Summary")
print("=" * 35)
print(f"  Batch Size    : {BATCH_SIZE}")
print(f"  Epochs        : {EPOCHS}")
print(f"  Learning Rate : {LEARNING_RATE}")
print(f"  Noise Factor  : {NOISE_FACTOR}")


## 3. Load & Prepare MNIST Dataset

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),                        # [0,255] → [0.0, 1.0]
])

# Download MNIST (train + test)
train_dataset = MNIST(root='./data', train=True,  download=True, transform=transform)
test_dataset  = MNIST(root='./data', train=False, download=True, transform=transform)

# Further split train → train / val
val_size   = 10_000
train_size = len(train_dataset) - val_size
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train samples : {len(train_dataset):,}")
print(f"Val   samples : {len(val_dataset):,}")
print(f"Test  samples : {len(test_dataset):,}")


## 4. Visualise Sample Images

In [ ]:
def add_noise(images, noise_factor=NOISE_FACTOR):
    """Add Gaussian noise and clamp to [0, 1]."""
    noise  = torch.randn_like(images) * noise_factor
    return torch.clamp(images + noise, 0., 1.)

# Grab one batch
sample_imgs, sample_labels = next(iter(test_loader))
noisy_imgs = add_noise(sample_imgs)

fig, axes = plt.subplots(2, 10, figsize=(18, 4))
fig.suptitle("Top row: Clean  |  Bottom row: Noisy (σ = {:.1f})".format(NOISE_FACTOR),
             fontsize=13, fontweight='bold', y=1.02)

for i in range(10):
    axes[0, i].imshow(sample_imgs[i].squeeze(), cmap='gray')
    axes[0, i].set_title(f"Label: {sample_labels[i].item()}", fontsize=8)
    axes[0, i].axis('off')

    axes[1, i].imshow(noisy_imgs[i].squeeze(), cmap='gray')
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Sample images plotted.")


## 5. Model Architecture — Convolutional Denoising Autoencoder

The autoencoder consists of two parts:

| Component | Layers | Output Shape |
|-----------|--------|-------------|
| **Encoder** | Conv2d(1→32) → BN → ReLU → MaxPool | 14×14 |
|  | Conv2d(32→64) → BN → ReLU → MaxPool | 7×7 |
|  | Conv2d(64→128) → BN → ReLU | 7×7 |
| **Decoder** | ConvTranspose2d(128→64) → BN → ReLU | 14×14 |
|  | ConvTranspose2d(64→32) → BN → ReLU | 28×28 |
|  | Conv2d(32→1) → Sigmoid | 28×28 |


In [ ]:
class DenoisingAutoencoder(nn.Module):
    """Convolutional Denoising Autoencoder for MNIST (28×28 grayscale)."""

    def __init__(self):
        super().__init__()

        # ── Encoder ──────────────────────────────────────────────────────────
        self.encoder = nn.Sequential(
            # Block 1: 28×28 → 14×14
            nn.Conv2d(1, 32, kernel_size=3, padding=1),   # (B, 32, 28, 28)
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                            # (B, 32, 14, 14)

            # Block 2: 14×14 → 7×7
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # (B, 64, 14, 14)
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                            # (B, 64,  7,  7)

            # Block 3: keep spatial size
            nn.Conv2d(64, 128, kernel_size=3, padding=1), # (B, 128, 7, 7)
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )

        # ── Decoder ──────────────────────────────────────────────────────────
        self.decoder = nn.Sequential(
            # Block 1: 7×7 → 14×14
            nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2),  # (B, 64, 14, 14)
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            # Block 2: 14×14 → 28×28
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),   # (B, 32, 28, 28)
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            # Output: keep spatial size, compress channels → 1
            nn.Conv2d(32, 1, kernel_size=3, padding=1),            # (B,  1, 28, 28)
            nn.Sigmoid(),                                           # pixel values in [0,1]
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)


model = DenoisingAutoencoder().to(DEVICE)
print(model)

# Parameter count
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable:,}")


## 6. Loss Function & Optimiser

In [ ]:
criterion = nn.MSELoss()                         # pixel-wise reconstruction loss
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, verbose=True
)

print("Loss      :", criterion)
print("Optimizer :", optimizer.__class__.__name__)
print("Scheduler :", scheduler.__class__.__name__)


## 7. Training the Autoencoder

In [ ]:
history = {'train_loss': [], 'val_loss': []}

def train_epoch(loader):
    model.train()
    running = 0.0
    for imgs, _ in loader:
        imgs = imgs.to(DEVICE)
        noisy = add_noise(imgs).to(DEVICE)

        optimizer.zero_grad()
        output = model(noisy)
        loss   = criterion(output, imgs)      # reconstruct CLEAN image
        loss.backward()
        optimizer.step()
        running += loss.item() * imgs.size(0)
    return running / len(loader.dataset)


def eval_epoch(loader):
    model.eval()
    running = 0.0
    with torch.no_grad():
        for imgs, _ in loader:
            imgs  = imgs.to(DEVICE)
            noisy = add_noise(imgs).to(DEVICE)
            output = model(noisy)
            loss   = criterion(output, imgs)
            running += loss.item() * imgs.size(0)
    return running / len(loader.dataset)


print(f"{'Epoch':>6} | {'Train Loss':>12} | {'Val Loss':>10} | {'LR':>10}")
print("-" * 50)

for epoch in range(1, EPOCHS + 1):
    t_loss = train_epoch(train_loader)
    v_loss = eval_epoch(val_loader)
    scheduler.step(v_loss)

    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)

    lr_now = optimizer.param_groups[0]['lr']
    print(f"{epoch:>6} | {t_loss:>12.6f} | {v_loss:>10.6f} | {lr_now:>10.2e}")

print("\n✅ Training complete.")


## 8. Training & Validation Loss Curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
epochs_range = range(1, EPOCHS + 1)

ax.plot(epochs_range, history['train_loss'], 'b-o', markersize=4, label='Train Loss')
ax.plot(epochs_range, history['val_loss'],   'r-s', markersize=4, label='Val Loss')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('MSE Loss', fontsize=12)
ax.set_title('Denoising Autoencoder — Loss Curves', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('loss_curves.png', dpi=120, bbox_inches='tight')
plt.show()


## 9. Qualitative Results — Visual Comparison

In [ ]:
model.eval()
test_imgs, test_labels = next(iter(test_loader))
noisy_test = add_noise(test_imgs)

with torch.no_grad():
    reconstructed = model(noisy_test.to(DEVICE)).cpu()

n = 10
fig = plt.figure(figsize=(20, 6))
gs  = gridspec.GridSpec(3, n, hspace=0.4, wspace=0.05)

row_titles = ["Original (Clean)", "Noisy Input", "Autoencoder Output (Denoised)"]
row_data   = [test_imgs, noisy_test, reconstructed]
cmaps      = ['gray', 'gray', 'gray']

for row_idx, (title, data) in enumerate(zip(row_titles, row_data)):
    for col_idx in range(n):
        ax = fig.add_subplot(gs[row_idx, col_idx])
        ax.imshow(data[col_idx].squeeze(), cmap='gray', vmin=0, vmax=1)
        ax.axis('off')
        if col_idx == 0:
            ax.set_ylabel(title, fontsize=9, rotation=90, labelpad=40, va='center')

fig.suptitle("Denoising Autoencoder — Reconstruction Results", fontsize=14, fontweight='bold', y=1.02)
plt.savefig('reconstruction_results.png', dpi=130, bbox_inches='tight')
plt.show()


## 10. Quantitative Evaluation — MSE & SSIM

In [ ]:
all_clean  = []
all_noisy  = []
all_recon  = []

model.eval()
with torch.no_grad():
    for imgs, _ in test_loader:
        noisy = add_noise(imgs)
        recon = model(noisy.to(DEVICE)).cpu()
        all_clean.append(imgs.numpy())
        all_noisy.append(noisy.numpy())
        all_recon.append(recon.numpy())

all_clean = np.concatenate(all_clean, axis=0).squeeze()   # (N, 28, 28)
all_noisy = np.concatenate(all_noisy, axis=0).squeeze()
all_recon = np.concatenate(all_recon, axis=0).squeeze()

# MSE
mse_noisy = mean_squared_error(all_clean.reshape(len(all_clean), -1),
                                all_noisy.reshape(len(all_noisy), -1),
                                multioutput='uniform_average')
mse_recon = mean_squared_error(all_clean.reshape(len(all_clean), -1),
                                all_recon.reshape(len(all_recon), -1),
                                multioutput='uniform_average')

# SSIM — averaged per image
ssim_noisy = np.mean([ssim(all_clean[i], all_noisy[i], data_range=1.0) for i in range(len(all_clean))])
ssim_recon = np.mean([ssim(all_clean[i], all_recon[i], data_range=1.0) for i in range(len(all_clean))])

print("Quantitative Metrics on Test Set")
print("=" * 45)
print(f"{'Metric':<20} {'Noisy vs Clean':>15} {'Denoised vs Clean':>18}")
print("-" * 45)
print(f"{'MSE  (↓ better)':<20} {mse_noisy:>15.5f} {mse_recon:>18.5f}")
print(f"{'SSIM (↑ better)':<20} {ssim_noisy:>15.4f} {ssim_recon:>18.4f}")
print("=" * 45)
improvement_mse  = (mse_noisy  - mse_recon)  / mse_noisy  * 100
improvement_ssim = (ssim_recon - ssim_noisy) / ssim_noisy  * 100
print(f"\n📉 MSE  improvement  : {improvement_mse:.1f}%")
print(f"📈 SSIM improvement  : {improvement_ssim:.1f}%")


## 11. Pixel-level Error Maps

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(16, 10))
fig.suptitle("Pixel-level Absolute Error Maps\n(darker = lower error)",
             fontsize=13, fontweight='bold')

for col in range(5):
    img_c = all_clean[col]
    img_n = all_noisy[col]
    img_r = all_recon[col]

    err_noisy = np.abs(img_c - img_n)
    err_recon = np.abs(img_c - img_r)

    axes[0, col].imshow(img_c, cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(f"Clean #{col}", fontsize=9)
    axes[0, col].axis('off')

    im1 = axes[1, col].imshow(err_noisy, cmap='hot', vmin=0, vmax=0.8)
    axes[1, col].set_title(f"Noisy err {err_noisy.mean():.3f}", fontsize=9)
    axes[1, col].axis('off')

    im2 = axes[2, col].imshow(err_recon, cmap='hot', vmin=0, vmax=0.8)
    axes[2, col].set_title(f"Denoised err {err_recon.mean():.3f}", fontsize=9)
    axes[2, col].axis('off')

fig.colorbar(im2, ax=axes[2, :], fraction=0.02, pad=0.02, label='Abs Error')
plt.tight_layout()
plt.savefig('error_maps.png', dpi=120, bbox_inches='tight')
plt.show()


## 12. Noise Robustness Analysis

In [ ]:
noise_levels = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
mse_results  = []
ssim_results = []

model.eval()
imgs_np = all_clean[:500]          # use 500 samples for speed

for nf in noise_levels:
    clean_t  = torch.tensor(imgs_np[:, np.newaxis, :, :])
    noisy_t  = torch.clamp(clean_t + torch.randn_like(clean_t) * nf, 0., 1.)
    with torch.no_grad():
        recon_t = model(noisy_t.to(DEVICE)).cpu()

    c = imgs_np
    r = recon_t.squeeze().numpy()
    mse_results.append(mean_squared_error(c.reshape(len(c),-1), r.reshape(len(r),-1)))
    ssim_results.append(np.mean([ssim(c[i], r[i], data_range=1.0) for i in range(len(c))]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(noise_levels, mse_results,  'b-o', markersize=6)
ax1.set_xlabel('Noise Factor (σ)', fontsize=11)
ax1.set_ylabel('Reconstruction MSE', fontsize=11)
ax1.set_title('MSE vs Noise Level', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.4)

ax2.plot(noise_levels, ssim_results, 'r-s', markersize=6)
ax2.set_xlabel('Noise Factor (σ)', fontsize=11)
ax2.set_ylabel('SSIM', fontsize=11)
ax2.set_title('SSIM vs Noise Level', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('noise_robustness.png', dpi=120, bbox_inches='tight')
plt.show()
print("Noise Level | MSE    | SSIM")
print("-"*32)
for nf, m, s in zip(noise_levels, mse_results, ssim_results):
    print(f"  σ = {nf:.1f}   | {m:.5f} | {s:.4f}")


## 13. Save the Trained Model

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'history': history,
    'hyperparams': {
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr': LEARNING_RATE,
        'noise_factor': NOISE_FACTOR,
    }
}, 'denoising_autoencoder_mnist.pth')

print("✅ Model saved → denoising_autoencoder_mnist.pth")

# Load & verify
checkpoint = torch.load('denoising_autoencoder_mnist.pth', map_location=DEVICE)
model_loaded = DenoisingAutoencoder().to(DEVICE)
model_loaded.load_state_dict(checkpoint['model_state_dict'])
model_loaded.eval()
print("✅ Model loaded and verified successfully.")


## 14. Conclusion & Key Takeaways

---

### ✅ Summary

| Step | Description |
|------|-------------|
| **Dataset** | MNIST — 70,000 grayscale 28×28 digit images |
| **Noise** | Gaussian noise added (σ = 0.4) to corrupt images |
| **Architecture** | 3-block Conv Encoder → 2-block ConvTranspose Decoder |
| **Loss** | Mean Squared Error (MSE) between output and clean image |
| **Optimizer** | Adam with ReduceLROnPlateau scheduler |
| **Evaluation** | Pixel-wise MSE, SSIM, visual inspection, noise robustness |

---

### 📌 Key Observations

1. **Effective Denoising:** The autoencoder significantly reduces pixel-level error — SSIM improves substantially over the noisy baseline.
2. **Bottleneck Representation:** The encoder forces the network to learn compact, meaningful features rather than memorising noise.
3. **Noise Robustness:** Performance degrades gracefully as noise level increases — the model generalises across different noise magnitudes.
4. **Batch Normalisation:** Stabilised training and improved convergence speed compared to networks without BN.

---

### 🔭 Possible Extensions

- **Variational Autoencoder (VAE):** Model uncertainty in the latent space
- **Adversarial Training (DAE + GAN):** Sharper, perceptually better reconstructions
- **U-Net Skip Connections:** Preserve fine spatial details during decoding
- **Real Noise Types:** Salt-and-pepper, Poisson, or motion-blur corruption
- **Transfer to Other Domains:** Apply the same pipeline to natural images (CIFAR-10, CelebA)
